## Import Libraries and Data

In [849]:
import pandas as pd
import numpy as np

In [850]:
df = pd.read_csv('cleaned_Melbourne-1.csv')
df

,Timestamp,Air Temp (degrees C),Dew Pt Temp (degrees C),Humidity (%),Wind Direction,Wind Speed (km/h),MSLP (hPa),Rainfall since 9 am (mm)
0,2011-01-01 00:04:00,24.8,14.0,51.0,SE,11.0,1007.4,0.0
1,2011-01-01 00:14:00,24.8,13.3,48.0,SE,11.0,1007.5,0.0
2,2011-01-01 00:24:00,24.9,13.3,48.0,SE,11.0,1007.5,0.0
3,2011-01-01 00:34:00,24.7,13.4,49.0,SE,11.0,1007.4,0.0
4,2011-01-01 00:44:00,24.1,13.3,51.0,ESE,9.0,1007.3,0.0
...,...,...,...,...,...,...,...,...
570573,2025-03-12 15:30:00,33.4,14.9,32.0,NW,20.0,1014.6,0.0
570574,2025-03-12 15:34:00,33.5,14.5,31.0,NNW,26.0,1014.6,0.0
570575,2025-03-12 16:00:00,34.4,14.8,30.0,WNW,26.0,1014.4,0.0
570576,2025-03-12 16:30:00,33.6,14.1,30.0,NW,28.0,1014.1,0.0


In [851]:
all_df = df

In [852]:
all_df.index = pd.to_datetime(all_df['Timestamp']) # Make sure the index is in Timsestamp format
all_df.drop(columns=['Timestamp'], inplace = True)
all_df.head()

,Air Temp (degrees C),Dew Pt Temp (degrees C),Humidity (%),Wind Direction,Wind Speed (km/h),MSLP (hPa),Rainfall since 9 am (mm)
Timestamp,,,,,,,
2011-01-01 00:04:00,24.8,14.0,51.0,SE,11.0,1007.4,0.0
2011-01-01 00:14:00,24.8,13.3,48.0,SE,11.0,1007.5,0.0
2011-01-01 00:24:00,24.9,13.3,48.0,SE,11.0,1007.5,0.0
2011-01-01 00:34:00,24.7,13.4,49.0,SE,11.0,1007.4,0.0
2011-01-01 00:44:00,24.1,13.3,51.0,ESE,9.0,1007.3,0.0


In [853]:
all_df.dtypes # Check all the data types

Air Temp (degrees C)        float64
Dew Pt Temp (degrees C)     float64
Humidity (%)                float64
Wind Direction               object
Wind Speed (km/h)           float64
MSLP (hPa)                  float64
Rainfall since 9 am (mm)    float64
dtype: object

In [854]:
# Convert all the missing values to NaN so it's easier to work with later on
all_df[['Wind Speed (km/h)', 'MSLP (hPa)']] = all_df[['Wind Speed (km/h)', 'MSLP (hPa)']].replace(-9999.0, np.nan)
all_df['Wind Direction'] = all_df['Wind Direction'].replace('-', np.nan)

In [855]:
all_df.isna().sum() # Check for this missing values

Air Temp (degrees C)          0
Dew Pt Temp (degrees C)       0
Humidity (%)                  0
Wind Direction               62
Wind Speed (km/h)            62
MSLP (hPa)                   34
Rainfall since 9 am (mm)    505
dtype: int64

Turn out that there are.....
- 62 missing values in both Wind Direction and Wind Speed (km/h)
- 34 missing values in MSLP (hPa)
- 505 missing values in Rainfall since 9 am (mm)

## Wind Speed & MSLP

First, we can deal with Wind Speed and MSLP by linear interpolation. Because these 2 columns contain numeric data which can be calculate and fill in those missing values easily.

In [856]:
all_df['Wind Speed (km/h)'] = all_df['Wind Speed (km/h)'].interpolate(method='linear')
all_df['MSLP (hPa)'] = all_df['MSLP (hPa)'].interpolate(method='linear')

In [857]:
columns = ['Wind Speed (km/h)', 'MSLP (hPa)']
all_df.isna().sum()

Air Temp (degrees C)          0
Dew Pt Temp (degrees C)       0
Humidity (%)                  0
Wind Direction               62
Wind Speed (km/h)             0
MSLP (hPa)                    0
Rainfall since 9 am (mm)    505
dtype: int64

So, no more missing values in Wind Speed and MSLP. Only Wind Direction and Rainfall left...

## Wind Direction

To deal with missing values in Wind Direction, we can do that by encoding the direction into xy-direction instead. With that, we will get numerical x and y data which represent all the direction.

- North = 90 degree = pi/2 radian => (x,y) = (0,1)
- East = 0 degree = 0 radian => (x,y) = (1,0)
- South = 270 degree = 3pi/4 radian => (x,y) = (0,-1)
- West = 180 degree = pi radian => (x,y) = (-1,0)

x = cos(radian), y = sin(radian)

In [858]:
all_df.loc[all_df['Wind Speed (km/h)'] == 0, 'Wind Direction'] = 'CALM'
all_df.isna().sum()

Air Temp (degrees C)          0
Dew Pt Temp (degrees C)       0
Humidity (%)                  0
Wind Direction               62
Wind Speed (km/h)             0
MSLP (hPa)                    0
Rainfall since 9 am (mm)    505
dtype: int64

### Encoded Wind Direction (xy-direction)

In [859]:
# Convert from string-direction to degree-direction (numeric)
direction_val = {
    'E':0, 'ENE':22.5, 'NE':45, 'NNE':67.5,
    'N':90, 'NNW':112.5, 'NW':135, 'WNW':157.5,
    'W':180, 'WSW':202.5, 'SW':225, 'SSW':247.5,
    'S':270, 'SSE':292.5, 'SE':315, 'ESE':337.5,
    'CALM':np.nan
}

all_df['Wind Direction (degree)'] = all_df['Wind Direction'].map(direction_val)
all_df.head()

,Air Temp (degrees C),Dew Pt Temp (degrees C),Humidity (%),Wind Direction,Wind Speed (km/h),MSLP (hPa),Rainfall since 9 am (mm),Wind Direction (degree)
Timestamp,,,,,,,,
2011-01-01 00:04:00,24.8,14.0,51.0,SE,11.0,1007.4,0.0,315.0
2011-01-01 00:14:00,24.8,13.3,48.0,SE,11.0,1007.5,0.0,315.0
2011-01-01 00:24:00,24.9,13.3,48.0,SE,11.0,1007.5,0.0,315.0
2011-01-01 00:34:00,24.7,13.4,49.0,SE,11.0,1007.4,0.0,315.0
2011-01-01 00:44:00,24.1,13.3,51.0,ESE,9.0,1007.3,0.0,337.5


In [860]:
# Convert degree-direction to radian-direction
all_df['Wind Direction (radian)'] = np.deg2rad(all_df['Wind Direction (degree)'])
all_df.head()

,Air Temp (degrees C),Dew Pt Temp (degrees C),Humidity (%),Wind Direction,Wind Speed (km/h),MSLP (hPa),Rainfall since 9 am (mm),Wind Direction (degree),Wind Direction (radian)
Timestamp,,,,,,,,,
2011-01-01 00:04:00,24.8,14.0,51.0,SE,11.0,1007.4,0.0,315.0,5.497787
2011-01-01 00:14:00,24.8,13.3,48.0,SE,11.0,1007.5,0.0,315.0,5.497787
2011-01-01 00:24:00,24.9,13.3,48.0,SE,11.0,1007.5,0.0,315.0,5.497787
2011-01-01 00:34:00,24.7,13.4,49.0,SE,11.0,1007.4,0.0,315.0,5.497787
2011-01-01 00:44:00,24.1,13.3,51.0,ESE,9.0,1007.3,0.0,337.5,5.890486


In [861]:
# Convert radian-direction to xy-direction (numeric)
all_df['Wind Direction (x)'] = np.sin(all_df['Wind Direction (radian)'])
all_df['Wind Direction (y)'] = np.cos(all_df['Wind Direction (radian)'])

# For CALM wind, we assign both xy-direction to 0.0 
# Because there is no wind which mean no wind direction at that period
all_df.loc[all_df['Wind Direction'] == 'CALM', ['Wind Direction (x)', 'Wind Direction (y)']] = 0.0

all_df.head()

,Air Temp (degrees C),Dew Pt Temp (degrees C),Humidity (%),Wind Direction,Wind Speed (km/h),MSLP (hPa),Rainfall since 9 am (mm),Wind Direction (degree),Wind Direction (radian),Wind Direction (x),Wind Direction (y)
Timestamp,,,,,,,,,,,
2011-01-01 00:04:00,24.8,14.0,51.0,SE,11.0,1007.4,0.0,315.0,5.497787,-0.707107,0.707107
2011-01-01 00:14:00,24.8,13.3,48.0,SE,11.0,1007.5,0.0,315.0,5.497787,-0.707107,0.707107
2011-01-01 00:24:00,24.9,13.3,48.0,SE,11.0,1007.5,0.0,315.0,5.497787,-0.707107,0.707107
2011-01-01 00:34:00,24.7,13.4,49.0,SE,11.0,1007.4,0.0,315.0,5.497787,-0.707107,0.707107
2011-01-01 00:44:00,24.1,13.3,51.0,ESE,9.0,1007.3,0.0,337.5,5.890486,-0.382683,0.923880


In [862]:
all_df.isna().sum()

Air Temp (degrees C)            0
Dew Pt Temp (degrees C)         0
Humidity (%)                    0
Wind Direction                 62
Wind Speed (km/h)               0
MSLP (hPa)                      0
Rainfall since 9 am (mm)      505
Wind Direction (degree)     16089
Wind Direction (radian)     16089
Wind Direction (x)             62
Wind Direction (y)             62
dtype: int64

Note that there are still missing values in Wind Direcion here since we haven't fixed that. But we can ignore those missing values in degree-direction and radian-direction as those missing values also include CALM wind.

So, that means there are 62 missing value rows in Wind Direction, not 16089 rows.

### Missing Values

Now, we can fill in those missing values of Wind Direction by operating linear interpolation on xy-direction (since these columns are already in numeric format).

In [863]:
all_df['Wind Direction (x)'] = all_df['Wind Direction (x)'].interpolate(method='linear')
all_df['Wind Direction (y)'] = all_df['Wind Direction (y)'].interpolate(method='linear')

In [864]:
# Drop these columns since we don't need them any more
all_df.drop(columns=['Wind Direction', 'Wind Direction (degree)', 'Wind Direction (radian)'], inplace=True)
all_df.head()

,Air Temp (degrees C),Dew Pt Temp (degrees C),Humidity (%),Wind Speed (km/h),MSLP (hPa),Rainfall since 9 am (mm),Wind Direction (x),Wind Direction (y)
Timestamp,,,,,,,,
2011-01-01 00:04:00,24.8,14.0,51.0,11.0,1007.4,0.0,-0.707107,0.707107
2011-01-01 00:14:00,24.8,13.3,48.0,11.0,1007.5,0.0,-0.707107,0.707107
2011-01-01 00:24:00,24.9,13.3,48.0,11.0,1007.5,0.0,-0.707107,0.707107
2011-01-01 00:34:00,24.7,13.4,49.0,11.0,1007.4,0.0,-0.707107,0.707107
2011-01-01 00:44:00,24.1,13.3,51.0,9.0,1007.3,0.0,-0.382683,0.923880


In [865]:
# We have to make sure that those interpolated values are valid and not out of the range [-1,1]
out_of_bounds_x = (all_df['Wind Direction (x)'] < -1.0) | (all_df['Wind Direction (x)'] > 1.0)
out_of_bounds_y = (all_df['Wind Direction (y)'] < -1.0) | (all_df['Wind Direction (y)'] > 1.0)

error = all_df[out_of_bounds_x | out_of_bounds_y] # Combine both out_of_bounds_x and out_of_bounds_y together
len(error) # print out the number of rows which are out of bounds

0

There is 0 data which are out of bounds. So, we can say that all the data in xy-direction that we just interpolated are all valid.

In [866]:
# Check for missing values
all_df.isna().sum()

Air Temp (degrees C)          0
Dew Pt Temp (degrees C)       0
Humidity (%)                  0
Wind Speed (km/h)             0
MSLP (hPa)                    0
Rainfall since 9 am (mm)    505
Wind Direction (x)            0
Wind Direction (y)            0
dtype: int64

And as we see, there are no more missing values in wind xy-direction. Thus, Wind Direcion data whihc is now in the form of xy-direction are all cleaned!!

## Rainfall

For rainfall, we first have to fill in those 505 missing values first, which can be done by linear interpolation as well. But before doing that, we have to make sure that the cumulative rainfall data are actually valid (no change or increasing till resetting point). There should be no decreasing rainfall within the same time period (9.01 AM till 9.00 AM of the next day)

In [867]:
# Create a rainfall day index to make it easier to work with
# Shift all the data back by 9 hours and 1 second to make 9AM become the midnight of that day
# That 1 second will make sure that 9AM will be included to the day before
# e.g. rainfall at 8.30 = 2, at 9.00 = 2, at 9.30 = 0
# This is because rainfall at 9.00 belongs to the cumulative rain from earlier till 9.00, which means the data of 9.00 belongs to the day before.

rainfall_days = (all_df.index - pd.Timedelta(hours=9, seconds=1)).date
all_df['rainfall_day_id'] = rainfall_days
all_df

,Air Temp (degrees C),Dew Pt Temp (degrees C),Humidity (%),Wind Speed (km/h),MSLP (hPa),Rainfall since 9 am (mm),Wind Direction (x),Wind Direction (y),rainfall_day_id
Timestamp,,,,,,,,,
2011-01-01 00:04:00,24.8,14.0,51.0,11.0,1007.4,0.0,-0.707107,0.707107,2010-12-31
2011-01-01 00:14:00,24.8,13.3,48.0,11.0,1007.5,0.0,-0.707107,0.707107,2010-12-31
2011-01-01 00:24:00,24.9,13.3,48.0,11.0,1007.5,0.0,-0.707107,0.707107,2010-12-31
2011-01-01 00:34:00,24.7,13.4,49.0,11.0,1007.4,0.0,-0.707107,0.707107,2010-12-31
2011-01-01 00:44:00,24.1,13.3,51.0,9.0,1007.3,0.0,-0.382683,0.923880,2010-12-31
...,...,...,...,...,...,...,...,...,...
2025-03-12 15:30:00,33.4,14.9,32.0,20.0,1014.6,0.0,0.707107,-0.707107,2025-03-12
2025-03-12 15:34:00,33.5,14.5,31.0,26.0,1014.6,0.0,0.923880,-0.382683,2025-03-12
2025-03-12 16:00:00,34.4,14.8,30.0,26.0,1014.4,0.0,0.382683,-0.923880,2025-03-12


In [868]:
# Use linear interpolation method to fill in those missing values
all_df['Rainfall since 9 am (mm)'] = all_df.groupby('rainfall_day_id')['Rainfall since 9 am (mm)'].transform(lambda x: x.interpolate(method='linear'))

But remember that this is cummulative rainfall, so we should check if rainfall data are increasing (or no change) with in the same day. If there are data which decreases during the same day, those data are invalid and we should fix that.

We can fix those data by replacing them with the highest rainfall of that day so far.

e.g. rainfall at 10.00 = 1.00, at 10.30 = 2.00 , at 11.00 = 1.5, at 11.30 = 1.5

then we should replace rainfall at 11.00 and 11.30 with 2.00 instead

In [871]:
# Check the maximum rainfall of that day so far
all_df['Daily_Max_So_Far'] = all_df.groupby('rainfall_day_id')['Rainfall since 9 am (mm)'].cummax()

# Extract all the invalid rows
error_rows = all_df['Rainfall since 9 am (mm)'] < all_df['Daily_Max_So_Far']
all_df[error_rows]

,Air Temp (degrees C),Dew Pt Temp (degrees C),Humidity (%),Wind Speed (km/h),MSLP (hPa),Rainfall since 9 am (mm),Wind Direction (x),Wind Direction (y),rainfall_day_id,Daily_Max_So_Far
Timestamp,,,,,,,,,,
2011-03-20 13:15:00,27.9,14.5,44.0,28.0,1009.1,0.0,0.707107,7.071068e-01,2011-03-20,0.2
2011-03-20 13:25:00,27.9,14.0,42.0,17.0,1008.9,0.0,0.923880,3.826834e-01,2011-03-20,0.2
2011-03-20 13:35:00,27.4,13.2,41.0,17.0,1008.8,0.0,0.707107,7.071068e-01,2011-03-20,0.2
2011-03-20 13:45:00,27.9,13.4,40.0,13.0,1008.6,0.0,0.707107,7.071068e-01,2011-03-20,0.2
2011-03-20 13:55:00,28.4,13.7,40.0,20.0,1008.5,0.0,0.923880,3.826834e-01,2011-03-20,0.2
...,...,...,...,...,...,...,...,...,...,...
2023-07-09 07:00:00,11.8,8.3,79.0,20.0,1013.8,0.0,1.000000,6.123234e-17,2023-07-08,0.2
2023-07-09 07:30:00,11.4,8.6,83.0,22.0,1014.3,0.0,0.923880,-3.826834e-01,2023-07-08,0.2
2023-07-09 08:00:00,12.0,8.5,79.0,20.0,1014.6,0.0,0.923880,-3.826834e-01,2023-07-08,0.2


In [872]:
# An example of invalid cummulative rainfall data
all_df.loc['2011-03-20 12:00':'2011-03-20 15:00']

,Air Temp (degrees C),Dew Pt Temp (degrees C),Humidity (%),Wind Speed (km/h),MSLP (hPa),Rainfall since 9 am (mm),Wind Direction (x),Wind Direction (y),rainfall_day_id,Daily_Max_So_Far
Timestamp,,,,,,,,,,
2011-03-20 12:05:00,26.2,13.5,45.0,24.0,1010.7,0.0,1.000000,6.123234e-17,2011-03-20,0.0
2011-03-20 12:15:00,26.9,13.5,43.0,20.0,1010.5,0.0,0.923880,3.826834e-01,2011-03-20,0.0
2011-03-20 12:25:00,26.9,13.5,43.0,26.0,1010.3,0.0,0.923880,3.826834e-01,2011-03-20,0.0
2011-03-20 12:35:00,26.1,13.8,46.0,30.0,1010.1,0.2,0.923880,3.826834e-01,2011-03-20,0.2
2011-03-20 12:45:00,27.3,14.3,45.0,22.0,1009.9,0.2,0.923880,3.826834e-01,2011-03-20,0.2
2011-03-20 12:55:00,27.2,13.7,43.0,26.0,1009.7,0.2,0.923880,3.826834e-01,2011-03-20,0.2
2011-03-20 13:05:00,27.3,14.3,45.0,28.0,1009.4,0.2,0.707107,7.071068e-01,2011-03-20,0.2
2011-03-20 13:15:00,27.9,14.5,44.0,28.0,1009.1,0.0,0.707107,7.071068e-01,2011-03-20,0.2
2011-03-20 13:25:00,27.9,14.0,42.0,17.0,1008.9,0.0,0.923880,3.826834e-01,2011-03-20,0.2


As you see, rainfall increases to 0.2 mm at 12:25 but then turned to 0.0 at 13:15 till 14:45. Therefore, we should replace those 0.0 with 0.2 which is the cummax value.

In [873]:
# Replacing those invalid cummulative data with cummax data
all_df['Rainfall since 9 am (mm)'] = all_df.groupby('rainfall_day_id')['Rainfall since 9 am (mm)'].cummax()

In [874]:
# Check with the same example
all_df.loc['2011-03-20 12:00':'2011-03-20 15:00']

,Air Temp (degrees C),Dew Pt Temp (degrees C),Humidity (%),Wind Speed (km/h),MSLP (hPa),Rainfall since 9 am (mm),Wind Direction (x),Wind Direction (y),rainfall_day_id,Daily_Max_So_Far
Timestamp,,,,,,,,,,
2011-03-20 12:05:00,26.2,13.5,45.0,24.0,1010.7,0.0,1.000000,6.123234e-17,2011-03-20,0.0
2011-03-20 12:15:00,26.9,13.5,43.0,20.0,1010.5,0.0,0.923880,3.826834e-01,2011-03-20,0.0
2011-03-20 12:25:00,26.9,13.5,43.0,26.0,1010.3,0.0,0.923880,3.826834e-01,2011-03-20,0.0
2011-03-20 12:35:00,26.1,13.8,46.0,30.0,1010.1,0.2,0.923880,3.826834e-01,2011-03-20,0.2
2011-03-20 12:45:00,27.3,14.3,45.0,22.0,1009.9,0.2,0.923880,3.826834e-01,2011-03-20,0.2
2011-03-20 12:55:00,27.2,13.7,43.0,26.0,1009.7,0.2,0.923880,3.826834e-01,2011-03-20,0.2
2011-03-20 13:05:00,27.3,14.3,45.0,28.0,1009.4,0.2,0.707107,7.071068e-01,2011-03-20,0.2
2011-03-20 13:15:00,27.9,14.5,44.0,28.0,1009.1,0.2,0.707107,7.071068e-01,2011-03-20,0.2
2011-03-20 13:25:00,27.9,14.0,42.0,17.0,1008.9,0.2,0.923880,3.826834e-01,2011-03-20,0.2


All the rows from 13:15 till 14:45 are now fill with 0.2 instead of 0.0.

This means all the Rainfall since 9 am (mm) data are all in a valid cummulative format (either no change or increasing, no decreasing).

Even after we had done interpolation, there are still 23 missing values. They could be the rows which are on the gap between days or could be something else. We should do some further investigation on that.

In [875]:
all_df.isna().sum()

Air Temp (degrees C)         0
Dew Pt Temp (degrees C)      0
Humidity (%)                 0
Wind Speed (km/h)            0
MSLP (hPa)                   0
Rainfall since 9 am (mm)    23
Wind Direction (x)           0
Wind Direction (y)           0
rainfall_day_id              0
Daily_Max_So_Far            23
dtype: int64

In [876]:
# Investigate those missing values
all_df[all_df['Rainfall since 9 am (mm)'].isna()]

,Air Temp (degrees C),Dew Pt Temp (degrees C),Humidity (%),Wind Speed (km/h),MSLP (hPa),Rainfall since 9 am (mm),Wind Direction (x),Wind Direction (y),rainfall_day_id,Daily_Max_So_Far
Timestamp,,,,,,,,,,
2024-08-30 12:00:00,18.2,1.3,32.0,30.0,997.6,NaN,3.826834e-01,-9.238795e-01,2024-08-30,NaN
2024-08-30 12:30:00,18.9,2.3,33.0,32.0,997.3,NaN,1.224647e-16,-1.000000e+00,2024-08-30,NaN
2024-08-30 12:30:00,18.9,2.3,33.0,30.0,997.3,NaN,3.826834e-01,-9.238795e-01,2024-08-30,NaN
2024-08-30 13:00:00,19.0,0.6,29.0,35.0,997.0,NaN,3.826834e-01,-9.238795e-01,2024-08-30,NaN
2024-08-30 13:30:00,19.4,-0.6,26.0,26.0,996.6,NaN,7.071068e-01,-7.071068e-01,2024-08-30,NaN
2024-08-30 13:30:00,19.4,-0.6,26.0,35.0,996.6,NaN,3.826834e-01,-9.238795e-01,2024-08-30,NaN
2024-08-30 14:00:00,19.5,1.5,30.0,22.0,996.6,NaN,9.238795e-01,-3.826834e-01,2024-08-30,NaN
2024-08-30 14:30:00,19.3,2.6,33.0,28.0,996.3,NaN,9.238795e-01,-3.826834e-01,2024-08-30,NaN
2024-08-30 15:00:00,18.7,2.1,33.0,30.0,996.4,NaN,7.071068e-01,-7.071068e-01,2024-08-30,NaN


We can see that these are all from the same period, 2024-08-30 12:00 till 2024-08-30 20:30.

To investigate more, we should check the rows earlier and after to see what's the pattern of the cummulative rainfall of this 2024-08-30 so far.

In [877]:
all_df.loc['2024-08-30 08:30':'2024-08-30 21:00']

,Air Temp (degrees C),Dew Pt Temp (degrees C),Humidity (%),Wind Speed (km/h),MSLP (hPa),Rainfall since 9 am (mm),Wind Direction (x),Wind Direction (y),rainfall_day_id,Daily_Max_So_Far
Timestamp,,,,,,,,,,
2024-08-30 08:30:00,15.5,7.0,57.0,19.0,998.4,4.0,1.224647e-16,-1.000000e+00,2024-08-29,4.0
2024-08-30 09:00:00,16.0,6.7,54.0,35.0,998.6,4.0,1.224647e-16,-1.000000e+00,2024-08-29,4.0
2024-08-30 12:00:00,18.2,1.3,32.0,30.0,997.6,NaN,3.826834e-01,-9.238795e-01,2024-08-30,NaN
2024-08-30 12:30:00,18.9,2.3,33.0,32.0,997.3,NaN,1.224647e-16,-1.000000e+00,2024-08-30,NaN
2024-08-30 12:30:00,18.9,2.3,33.0,30.0,997.3,NaN,3.826834e-01,-9.238795e-01,2024-08-30,NaN
2024-08-30 13:00:00,19.0,0.6,29.0,35.0,997.0,NaN,3.826834e-01,-9.238795e-01,2024-08-30,NaN
2024-08-30 13:30:00,19.4,-0.6,26.0,26.0,996.6,NaN,7.071068e-01,-7.071068e-01,2024-08-30,NaN
2024-08-30 13:30:00,19.4,-0.6,26.0,35.0,996.6,NaN,3.826834e-01,-9.238795e-01,2024-08-30,NaN
2024-08-30 14:00:00,19.5,1.5,30.0,22.0,996.6,NaN,9.238795e-01,-3.826834e-01,2024-08-30,NaN


At 08:30 and 09:00 the rain were 4.0 mm, but that don't tell us any further information since these 2 rows belong to the day before (2024-08-29).

But as we see, there are rows missing between 9 AM to 12 PM on 2024-08-30. We might have to deal with this later when we do time interval.

If we scroll to 2024-08-30 21:00:00, we will see that the cummulative rainfall at that time was 0.0, that means there was no rain since 9 AM of 2024-08-30 till that time. Therefore, we can simply fill in those NaN with 0.0.

Note that this is probably not the best approach, but it works in this case.

In [878]:
all_df['Rainfall since 9 am (mm)'] = all_df['Rainfall since 9 am (mm)'].fillna(0) # Fill in those NaN with 0.0
all_df.loc['2024-08-30 08:30':'2024-08-30 21:00']

,Air Temp (degrees C),Dew Pt Temp (degrees C),Humidity (%),Wind Speed (km/h),MSLP (hPa),Rainfall since 9 am (mm),Wind Direction (x),Wind Direction (y),rainfall_day_id,Daily_Max_So_Far
Timestamp,,,,,,,,,,
2024-08-30 08:30:00,15.5,7.0,57.0,19.0,998.4,4.0,1.224647e-16,-1.000000e+00,2024-08-29,4.0
2024-08-30 09:00:00,16.0,6.7,54.0,35.0,998.6,4.0,1.224647e-16,-1.000000e+00,2024-08-29,4.0
2024-08-30 12:00:00,18.2,1.3,32.0,30.0,997.6,0.0,3.826834e-01,-9.238795e-01,2024-08-30,NaN
2024-08-30 12:30:00,18.9,2.3,33.0,32.0,997.3,0.0,1.224647e-16,-1.000000e+00,2024-08-30,NaN
2024-08-30 12:30:00,18.9,2.3,33.0,30.0,997.3,0.0,3.826834e-01,-9.238795e-01,2024-08-30,NaN
2024-08-30 13:00:00,19.0,0.6,29.0,35.0,997.0,0.0,3.826834e-01,-9.238795e-01,2024-08-30,NaN
2024-08-30 13:30:00,19.4,-0.6,26.0,26.0,996.6,0.0,7.071068e-01,-7.071068e-01,2024-08-30,NaN
2024-08-30 13:30:00,19.4,-0.6,26.0,35.0,996.6,0.0,3.826834e-01,-9.238795e-01,2024-08-30,NaN
2024-08-30 14:00:00,19.5,1.5,30.0,22.0,996.6,0.0,9.238795e-01,-3.826834e-01,2024-08-30,NaN


In [879]:
# Chech for missing values
all_df.drop(columns=['Daily_Max_So_Far'], inplace=True)
all_df.isna().sum()

Air Temp (degrees C)        0
Dew Pt Temp (degrees C)     0
Humidity (%)                0
Wind Speed (km/h)           0
MSLP (hPa)                  0
Rainfall since 9 am (mm)    0
Wind Direction (x)          0
Wind Direction (y)          0
rainfall_day_id             0
dtype: int64

There is no more missing values!!!!

Now, we should convert cumulative rainfall to period rainfall.

### Rainfall (Period)

In [880]:
# Get period rainfall by calculating the difference between the row and the row before
all_df['Period_Rainfall'] = all_df.groupby('rainfall_day_id')['Rainfall since 9 am (mm)'].diff()
all_df

,Air Temp (degrees C),Dew Pt Temp (degrees C),Humidity (%),Wind Speed (km/h),MSLP (hPa),Rainfall since 9 am (mm),Wind Direction (x),Wind Direction (y),rainfall_day_id,Period_Rainfall
Timestamp,,,,,,,,,,
2011-01-01 00:04:00,24.8,14.0,51.0,11.0,1007.4,0.0,-0.707107,0.707107,2010-12-31,NaN
2011-01-01 00:14:00,24.8,13.3,48.0,11.0,1007.5,0.0,-0.707107,0.707107,2010-12-31,0.0
2011-01-01 00:24:00,24.9,13.3,48.0,11.0,1007.5,0.0,-0.707107,0.707107,2010-12-31,0.0
2011-01-01 00:34:00,24.7,13.4,49.0,11.0,1007.4,0.0,-0.707107,0.707107,2010-12-31,0.0
2011-01-01 00:44:00,24.1,13.3,51.0,9.0,1007.3,0.0,-0.382683,0.923880,2010-12-31,0.0
...,...,...,...,...,...,...,...,...,...,...
2025-03-12 15:30:00,33.4,14.9,32.0,20.0,1014.6,0.0,0.707107,-0.707107,2025-03-12,0.0
2025-03-12 15:34:00,33.5,14.5,31.0,26.0,1014.6,0.0,0.923880,-0.382683,2025-03-12,0.0
2025-03-12 16:00:00,34.4,14.8,30.0,26.0,1014.4,0.0,0.382683,-0.923880,2025-03-12,0.0


In [881]:
# Check for missing values
all_df.isna().sum()

Air Temp (degrees C)           0
Dew Pt Temp (degrees C)        0
Humidity (%)                   0
Wind Speed (km/h)              0
MSLP (hPa)                     0
Rainfall since 9 am (mm)       0
Wind Direction (x)             0
Wind Direction (y)             0
rainfall_day_id                0
Period_Rainfall             5181
dtype: int64

These missing values are due to the reset period.

When calculating the differences, python will compare the data with the row before. That means for every 'start' row (in this case is every first row after 9 AM) will have no 'before' row to compare with. Thus, python will fill that with NaN.

To fix this, we can just simply fill in the actual data from Rainfall since 9 am (mm) to those rows.

In [882]:
# Further investigate if those rows are actually the first row after 9 AM.
all_df[all_df['Period_Rainfall'].isna()]

,Air Temp (degrees C),Dew Pt Temp (degrees C),Humidity (%),Wind Speed (km/h),MSLP (hPa),Rainfall since 9 am (mm),Wind Direction (x),Wind Direction (y),rainfall_day_id,Period_Rainfall
Timestamp,,,,,,,,,,
2011-01-01 00:04:00,24.8,14.0,51.0,11.0,1007.4,0.0,-0.707107,7.071068e-01,2010-12-31,NaN
2011-01-01 09:04:00,17.7,11.7,68.0,13.0,1012.5,0.0,-0.707107,-7.071068e-01,2011-01-01,NaN
2011-01-02 09:14:00,17.1,9.4,60.0,24.0,1016.5,0.0,-0.923880,-3.826834e-01,2011-01-02,NaN
2011-01-03 09:04:00,15.4,4.2,47.0,26.0,1018.1,0.0,-1.000000,-1.836970e-16,2011-01-03,NaN
2011-01-04 09:04:00,15.1,7.7,61.0,17.0,1014.1,0.0,-1.000000,-1.836970e-16,2011-01-04,NaN
...,...,...,...,...,...,...,...,...,...,...
2025-03-08 09:30:00,23.6,13.2,52.0,39.0,1020.8,0.0,1.000000,6.123234e-17,2025-03-08,NaN
2025-03-09 09:30:00,24.1,13.6,52.0,32.0,1018.7,0.0,0.923880,3.826834e-01,2025-03-09,NaN
2025-03-10 09:30:00,24.3,18.7,71.0,17.0,1018.0,0.0,0.923880,3.826834e-01,2025-03-10,NaN


In [883]:
# Replace NaNs with the actual data from Rainfall since 9 am (mm) column
all_df['Period_Rainfall'] = all_df['Period_Rainfall'].fillna(all_df['Rainfall since 9 am (mm)'])
all_df

,Air Temp (degrees C),Dew Pt Temp (degrees C),Humidity (%),Wind Speed (km/h),MSLP (hPa),Rainfall since 9 am (mm),Wind Direction (x),Wind Direction (y),rainfall_day_id,Period_Rainfall
Timestamp,,,,,,,,,,
2011-01-01 00:04:00,24.8,14.0,51.0,11.0,1007.4,0.0,-0.707107,0.707107,2010-12-31,0.0
2011-01-01 00:14:00,24.8,13.3,48.0,11.0,1007.5,0.0,-0.707107,0.707107,2010-12-31,0.0
2011-01-01 00:24:00,24.9,13.3,48.0,11.0,1007.5,0.0,-0.707107,0.707107,2010-12-31,0.0
2011-01-01 00:34:00,24.7,13.4,49.0,11.0,1007.4,0.0,-0.707107,0.707107,2010-12-31,0.0
2011-01-01 00:44:00,24.1,13.3,51.0,9.0,1007.3,0.0,-0.382683,0.923880,2010-12-31,0.0
...,...,...,...,...,...,...,...,...,...,...
2025-03-12 15:30:00,33.4,14.9,32.0,20.0,1014.6,0.0,0.707107,-0.707107,2025-03-12,0.0
2025-03-12 15:34:00,33.5,14.5,31.0,26.0,1014.6,0.0,0.923880,-0.382683,2025-03-12,0.0
2025-03-12 16:00:00,34.4,14.8,30.0,26.0,1014.4,0.0,0.382683,-0.923880,2025-03-12,0.0


In [884]:
# Check for missing values
all_df.isna().sum()

Air Temp (degrees C)        0
Dew Pt Temp (degrees C)     0
Humidity (%)                0
Wind Speed (km/h)           0
MSLP (hPa)                  0
Rainfall since 9 am (mm)    0
Wind Direction (x)          0
Wind Direction (y)          0
rainfall_day_id             0
Period_Rainfall             0
dtype: int64

In [885]:
# Drop unwantted columns
all_df.drop(columns=['rainfall_day_id','Rainfall since 9 am (mm)'], inplace=True)
all_df.head()

,Air Temp (degrees C),Dew Pt Temp (degrees C),Humidity (%),Wind Speed (km/h),MSLP (hPa),Wind Direction (x),Wind Direction (y),Period_Rainfall
Timestamp,,,,,,,,
2011-01-01 00:04:00,24.8,14.0,51.0,11.0,1007.4,-0.707107,0.707107,0.0
2011-01-01 00:14:00,24.8,13.3,48.0,11.0,1007.5,-0.707107,0.707107,0.0
2011-01-01 00:24:00,24.9,13.3,48.0,11.0,1007.5,-0.707107,0.707107,0.0
2011-01-01 00:34:00,24.7,13.4,49.0,11.0,1007.4,-0.707107,0.707107,0.0
2011-01-01 00:44:00,24.1,13.3,51.0,9.0,1007.3,-0.382683,0.923880,0.0


## No more MISSING VALUES!!!!

We have succesfully fill in all those missing values. But remember, there are MISSING ROWS too...TT

So, for now, we can download this data and try to convert them into a consistent time interval first (30-min and 1-hour interval). After that, we can check again if there are missing rows that need to be working on.

Nice Job!

In [819]:
# Downloading all_df as csv file
# You guys can name the file the way you want

# all_df.to_csv('cleaned_Melbourne_no_missingvalues-1.csv')